# Proyecto de Sistemas de Recomendacion

In [1]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score
import numpy as np

## 1) Cargado de datos

In [2]:
# Cargar y limpiar los datos
df = pd.read_csv("https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv")
df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)

In [3]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K


In [4]:
df.info()

<class 'pandas.DataFrame'>
Index: 30162 entries, 1 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             30162 non-null  int64
 1   workclass       30162 non-null  str  
 2   fnlwgt          30162 non-null  int64
 3   education       30162 non-null  str  
 4   education.num   30162 non-null  int64
 5   marital.status  30162 non-null  str  
 6   occupation      30162 non-null  str  
 7   relationship    30162 non-null  str  
 8   race            30162 non-null  str  
 9   sex             30162 non-null  str  
 10  capital.gain    30162 non-null  int64
 11  capital.loss    30162 non-null  int64
 12  hours.per.week  30162 non-null  int64
 13  native.country  30162 non-null  str  
 14  income          30162 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [5]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,30162.000000,3.016200e+04,30162.000000,30162.000000,30162.000000,30162.000000
mean,38.437902,1.897938e+05,10.121312,1092.007858,88.372489,40.931238
std,13.134665,1.056530e+05,2.549995,7406.346497,404.298370,11.979984
min,17.000000,1.376900e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.176272e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.784250e+05,10.000000,0.000000,0.000000,40.000000
75%,47.000000,2.376285e+05,13.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [6]:
#Variable Objetivo
df['income'].value_counts()

income
<=50K    22654
>50K      7508
Name: count, dtype: int64

In [7]:
(7508/(22654+7508))*100

24.892248524633644

## 2) Definición del Problema de Recomendación

* ¿Qué se quiere recomendar? Se busca recomendar si un perfil de vida actual (combinación de educación, ocupación y esfuerzo laboral) es conducente a un nivel de ingresos superior (>50K). En un contexto de "trayectorias", el sistema evalúa qué tan "exitosa" financieramente es la configuración actual del usuario.

* ¿Cuál será el "usuario" en este caso? El usuario se define como una entidad caracterizada por sus atributos socio-demográficos y laborales presentes en el censo.

* ¿Qué variables definen el perfil de un usuario? Según tu código, el perfil se define por: `age`, `education`, `marital.status`, `occupation`, `hours.per.week`, `sex` y `native.country`.

## 3) Preprocesamiento de datos

In [8]:
df['income'] = df['income'].str.strip() # elimina espacio en blanco en el texto
df['high_income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)

In [9]:
#Variable Objetivo
df['high_income'].value_counts()

high_income
0    22654
1     7508
Name: count, dtype: int64

In [10]:
#Representación del usuario
# Seleccionamos las variables que describen al usuario, igual que en un recomendador basado en contenido.

features = [
    "age",
    "education",
    "marital.status",
    "occupation",
    "hours.per.week",
    "sex",
    "native.country"
]

X = df[features]
y = df["high_income"]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [12]:
numeric_features = ["age", "hours.per.week"]
categorical_features = [
    "education",
    "marital.status",
    "occupation",
    "sex",
    "native.country"
]

In [13]:
# Aplicar get_dummies SOLO sobre train
X_train_processed = pd.get_dummies(X_train, columns=categorical_features)

# Aplicar lo mismo a test
X_test_processed = pd.get_dummies(X_test, columns=categorical_features)

# Alinear columnas 
X_train_processed, X_test_processed = X_train_processed.align(
    X_test_processed, 
    join="left", 
    axis=1, 
    fill_value=0
)

## 5) Construcción del Sistema de Recomendacion (Filtrado Basado en Contenido)

In [14]:
# =========================
# 5. Modelo XGBoost
# =========================
model = XGBClassifier(
    eval_metric="auc",
    random_state=42
)

model.fit(X_train_processed, y_train)

# =========================
# 6. Evaluación con AUC
# =========================
y_probs = model.predict_proba(X_test_processed)[:, 1]
auc = roc_auc_score(y_test, y_probs)

print(f"AUC en test: {auc:.4f}")

# =========================
# 7. Mejor threshold con F1
# =========================
thresholds = np.linspace(0.01, 0.99, 99)
f1_scores = []

for threshold in thresholds:
    y_pred = (y_probs >= threshold).astype(int)
    f1 = f1_score(y_test, y_pred)
    f1_scores.append(f1)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"Mejor threshold según F1: {best_threshold:.2f}")
print(f"Mejor F1-score: {best_f1:.4f}")

AUC en test: 0.8972
Mejor threshold según F1: 0.34
Mejor F1-score: 0.6898


In [15]:
# Crear artefacto
artifact = {
    "model": model,
    "feature_columns": X_train_processed.columns.tolist(),
    "best_threshold": float(best_threshold)
}

In [16]:
artifact

{'model': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=False, eval_metric='auc', feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...),
 'feature_columns': ['age',
  'hours.per.week',
  'education_10th',
  'education_11th',
  'education_12th',
  'education_1st-4th',
  'education_5th-6th',
  'education_7th-8th',
  'education_9th',
  'education_Assoc-ac

In [17]:
# Guardar
with open("xgb_income_recommender.pkl", "wb") as f:
    pickle.dump(artifact, f)

print("Modelo guardado con pickle")

Modelo guardado con pickle


In [18]:
def recommend_trajectory_production(user_profile, model_path="xgb_income_recommender.pkl"):
    """
    user_profile: dict con las claves:
    - age
    - education
    - marital.status
    - occupation
    - hours.per.week
    - sex
    - native.country
    """

    # 1. Cargar artefacto
    with open(model_path, "rb") as f:
        artifact = pickle.load(f)

    model = artifact["model"]
    feature_columns = artifact["feature_columns"]
    best_threshold = artifact["best_threshold"]

    # 2. Convertir a DataFrame
    user_df = pd.DataFrame([user_profile])

    # 3. get_dummies
    user_processed = pd.get_dummies(user_df)

    # 4. Alinear columnas (CLAVE en producción)***************************************************************************************
    user_processed = user_processed.reindex(
        columns=feature_columns,
        fill_value=0
    )

    # 5. Score
    prob = model.predict_proba(user_processed)[0][1]

    # 6. Decisión con threshold óptimo
    prediction = 1 if prob >= best_threshold else 0

    # 7. Output
    return {
        "probability": float(prob),
        "prediction": int(prediction),
        "threshold": float(best_threshold),
        "message": (
            f"✅ Probabilidad de >50K: {prob*100:.1f}%"
            if prediction == 1
            else f"⚠️ Probabilidad baja ({prob*100:.1f}%)"
        )
    }

In [19]:
simulated_user = {
    "age": 25,
    "education": "HS-grad",
    "marital.status": "Never-married",
    "occupation": "Sales",
    "hours.per.week": 30,
    "sex": "Male",
    "native.country": "United-States"
}

resultado = recommend_trajectory_production(simulated_user)
print(resultado)

{'probability': 0.004391741938889027, 'prediction': 0, 'threshold': 0.34, 'message': '⚠️ Probabilidad baja (0.4%)'}


El user simulado presenta una baja probabilidad de tener un ingreso mayor a los 50k anuales.